[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_03/12_lineas_de_transmision.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 12 — Líneas de transmisión

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 3**

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Obtener $C'$, $L'$, $Z_0$ y $u_p$ de un cable coaxial a partir de su
   geometría.
2. Calcular el coeficiente de reflexión de una carga cualquiera.
3. Usar la fórmula de impedancia de entrada para ver qué impedancia
   "aparece" al otro extremo de la línea.
4. Explicar por qué una línea transforma impedancias y cada cuánto se repite
   el patrón.

In [ ]:
# Preparación del entorno: funciona igual en Google Colab y en una copia local.
import sys
import urllib.request
from pathlib import Path

MODULOS = ["utilidades_notebook.py", "constantes_fisicas.py", "campos_electrostaticos.py", "lineas_transmision.py"]
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)

# Se exige que estén *todos* los módulos, no solo la carpeta: así, si otro
# notebook ya creó `src/` en esta misma sesión, igual se descarga lo que falte.
raiz = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if all((p / "src" / modulo).exists() for modulo in MODULOS)),
    None,
)
if raiz is None:  # Google Colab: descargar los módulos del curso.
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo in MODULOS:
        if not (raiz / "src" / modulo).exists():
            urllib.request.urlretrieve(URL_SRC + modulo, raiz / "src" / modulo)
sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from campos_electrostaticos import capacitancia_coaxial, inductancia_coaxial
from lineas_transmision import (
    coeficiente_reflexion,
    impedancia_entrada,
    longitud_electrica,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. De dónde sale todo esto

### 2.1 Cuándo un cable deja de ser un cable

En circuitos usted supone que el voltaje es el mismo en los dos extremos de
un cable. Eso vale mientras el cable sea mucho más corto que la longitud de
onda de la señal.

Cuando deja de valer, hay que aceptar que el voltaje y la corriente son
**funciones de la posición**. El cable pasa a ser una línea de transmisión y
se modela como una cadena infinita de pequeños $L$ y $C$ distribuidos.

La regla práctica: si el cable mide más de una décima de longitud de onda,
trátelo como línea.

### 2.2 Los parámetros son los de la Unidad 1

Aquí hay algo que vale la pena notar. $C'$ y $L'$ de un coaxial son
exactamente los que usted calculó en la semana 3 con electrostática y
magnetostática. La estructura es la misma; lo que cambió es la pregunta.

Por eso este notebook importa las funciones desde
`campos_electrostaticos.py`: no hay fórmulas nuevas que aprender.

### 2.3 Qué es la impedancia de entrada

Si usted conecta una carga $Z_L$ al final de una línea y mide con un
instrumento en la entrada, **no mide $Z_L$**. Mide otra cosa, que depende de
la longitud de la línea.

La razón es que a la entrada llegan dos ondas: la que va y la que vuelve
reflejada en la carga. Su suma depende de la fase relativa, y esa fase
depende de cuánto camino recorrieron.

## 3. Ecuaciones

**Parámetros por unidad de longitud de un coaxial de radios $a < b$:**

$$
C' = \frac{2\pi\varepsilon_0\varepsilon_r}{\ln(b/a)},
\qquad
L' = \frac{\mu_0\ln(b/a)}{2\pi}.
$$

**Impedancia característica y velocidad:**

$$
Z_0 = \sqrt{\frac{L'}{C'}},
\qquad
u_p = \frac{1}{\sqrt{L'C'}},
\qquad
\beta = \frac{\omega}{u_p} = \frac{2\pi}{\lambda}.
$$

**Coeficiente de reflexión en la carga:**

$$
\Gamma = \frac{Z_L - Z_0}{Z_L + Z_0}.
$$

Vale cero si $Z_L = Z_0$: la carga adaptada no refleja nada.

**Impedancia de entrada de una línea sin pérdidas de longitud $l$:**

$$
Z_{\text{in}} = Z_0\,
\frac{Z_L + jZ_0\tan(\beta l)}{Z_0 + jZ_L\tan(\beta l)} .
$$

Como $\tan$ tiene período $\pi$, la impedancia de entrada **se repite cada
media longitud de onda**.

## 4. Qué significa físicamente

**$Z_0$ no es una resistencia que disipe.** Aunque se mide en ohms, no
representa nada que se caliente. Es el cociente entre voltaje y corriente de
una onda que viaja: la línea "se siente" como esa resistencia mientras la
onda no haya llegado al otro extremo.

**Adaptar significa no reflejar.** Si $Z_L = Z_0$, la onda llega a la carga y
se entrega completa. Cualquier otro valor devuelve parte de la energía, y esa
energía que vuelve puede dañar el transmisor.

**La línea transforma impedancias.** Una carga de $100 - 25j~\Omega$ vista a
través de 0.15 longitudes de onda parece otra cosa completamente distinta.
Esto no es un defecto: es la herramienta que se usa para adaptar, como
veremos en la semana 13.

**Todo se repite cada media longitud de onda.** Recorrer $\lambda/2$ agrega
$\pi$ radianes a $\beta l$, y la tangente vuelve al mismo valor. Por eso al
trabajar con líneas solo importa la longitud **módulo** $\lambda/2$.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: el cable coaxial ---
radio_interno = 0.0005   # radio del conductor interior a [m]
radio_externo = 0.002    # radio del conductor exterior b [m]
eps_r = 2.25             # permitividad relativa del aislante
frecuencia = 1.0e9       # frecuencia de trabajo [Hz]

# --- Problema 2: línea con carga ---
Z0 = 50.0                    # impedancia característica de la línea [ohm]
ZL = 100.0 - 25.0j           # impedancia de la carga [ohm]
longitud_sobre_lambda = 0.150  # longitud de la línea, en longitudes de onda

## 6. Implementación

### 6.1 Problema 1 — parámetros del coaxial

In [ ]:
C_por_metro = capacitancia_coaxial(radio_interno, radio_externo, eps_r)
L_por_metro = inductancia_coaxial(radio_interno, radio_externo)

Z0_coaxial = np.sqrt(L_por_metro / C_por_metro)
velocidad = 1.0 / np.sqrt(L_por_metro * C_por_metro)
beta = 2.0 * np.pi * frecuencia / velocidad
longitud_onda = velocidad / frecuencia

### 6.2 Problema 2 — reflexión e impedancia de entrada

In [ ]:
Gamma = coeficiente_reflexion(ZL, Z0)
beta_l = longitud_electrica(longitud_sobre_lambda)
Zin = impedancia_entrada(ZL, Z0, beta_l)

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [
        ("Capacitancia por metro", "C'", C_por_metro, "F/m"),
        ("Inductancia por metro", "L'", L_por_metro, "H/m"),
        ("Impedancia característica", "Z_0", Z0_coaxial, "ohm"),
        ("Velocidad de propagación", "u_p", velocidad, "m/s"),
        ("Fracción de la velocidad de la luz", "u_p/c", velocidad / 2.99792458e8, "-"),
        ("Longitud de onda en el cable", "lambda", longitud_onda, "m"),
        ("Constante de fase", "beta", beta, "rad/m"),
    ]
)

In [ ]:
tabla_resultados(
    [
        ("Reflexión, parte real", "Re(Gamma)", Gamma.real, "-"),
        ("Reflexión, parte imaginaria", "Im(Gamma)", Gamma.imag, "-"),
        ("Módulo de la reflexión", "|Gamma|", abs(Gamma), "-"),
        ("Fase de la reflexión", "arg(Gamma)", np.rad2deg(np.angle(Gamma)), "grados"),
        ("Impedancia de entrada, parte real", "Re(Z_in)", Zin.real, "ohm"),
        ("Impedancia de entrada, parte imaginaria", "Im(Z_in)", Zin.imag, "ohm"),
        ("Potencia reflejada", "|Gamma|^2", abs(Gamma) ** 2, "-"),
    ]
)

Comprobación de la periodicidad: la impedancia de entrada a $l$ y a
$l + \lambda/2$ debe ser la misma.

In [ ]:
Zin_media_onda_despues = impedancia_entrada(
    ZL, Z0, longitud_electrica(longitud_sobre_lambda + 0.5)
)
print(f"Z_in a {longitud_sobre_lambda:.3f} lambda        = {Zin:.6f} ohm")
print(f"Z_in a {longitud_sobre_lambda + 0.5:.3f} lambda        = {Zin_media_onda_despues:.6f} ohm")
print(f"Diferencia = {abs(Zin - Zin_media_onda_despues):.3e} ohm")

## 8. Visualización

Cómo cambia la impedancia de entrada según la longitud de la línea. Fíjese en
que el patrón se repite cada media longitud de onda.

In [ ]:
longitudes = np.linspace(0.0, 1.0, 800)
Zin_curva = impedancia_entrada(ZL, Z0, longitud_electrica(longitudes))

fig, eje = plt.subplots()
eje.plot(longitudes, Zin_curva.real, label="parte real")
eje.plot(longitudes, Zin_curva.imag, label="parte imaginaria")
eje.axhline(Z0, color="black", linestyle=":", label=f"Z_0 = {Z0:.0f} ohm")
for repeticion in (0.5, 1.0):
    eje.axvline(repeticion, color="gray", linestyle="--", linewidth=0.8)
eje.scatter([longitud_sobre_lambda], [Zin.real], color="tab:blue", zorder=5)
eje.scatter([longitud_sobre_lambda], [Zin.imag], color="tab:orange", zorder=5)
eje.set_xlabel("Longitud de la línea, en longitudes de onda")
eje.set_ylabel("Impedancia (ohm)")
eje.set_title("La línea transforma la impedancia de la carga")
eje.legend()
fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**El cable resultó de unos 55 $\Omega$.** Está cerca de los 50 $\Omega$
estándar. Para llegar exactamente a 50 habría que ajustar la relación $b/a$,
que es justo lo que hacen los fabricantes.

**La señal viaja a dos tercios de $c$.** Con $\varepsilon_r = 2.25$,
$u_p = c/1.5$. Ese número se llama "factor de velocidad" y aparece en las
hojas de datos de los cables: aquí es 0.667.

**La carga refleja un 13 % de la potencia.** Con $|\Gamma| \approx 0.368$,
la potencia devuelta es $|\Gamma|^2 \approx 0.135$. No es adaptación, pero
tampoco es catastrófico.

**La impedancia de entrada no se parece a la carga.** Partiendo de
$100 - 25j$, a 0.15 longitudes de onda se ve algo así como $28 - 19j$. Las
líneas verticales del gráfico marcan dónde el patrón vuelve a empezar: la
comprobación numérica da una diferencia del orden de $10^{-14}~\Omega$.

**La curva cruza $Z_0$ dos veces por período.** En esos puntos la impedancia
de entrada es real. Esa observación es la base del método de adaptación con
stub que se estudia con la carta de Smith.

## 10. Ejercicios para experimentar

            1. Ponga `ZL = 50.0`, igual a `Z0`. ¿Cuánto vale $\Gamma$? ¿Cómo se ve el
               gráfico? ¿Por qué la línea deja de transformar?
            2. Ponga `ZL = 0.0` (cortocircuito). ¿Cuánto vale $\Gamma$? ¿Qué impedancia
               ve a un cuarto de longitud de onda? Compare con un circuito abierto.
            3. Ponga `longitud_sobre_lambda = 0.25`. Verifique que se cumple
               $Z_{\text{in}} = Z_0^2 / Z_L$.
            4. Ponga `longitud_sobre_lambda = 0.5`. ¿Qué obtiene? ¿Por qué media longitud
               de onda "no hace nada"?
            5. Cambie `eps_r` a `1.0` (coaxial con aire). ¿Cómo cambian $Z_0$ y $u_p$?
            6. Ajuste `radio_externo` hasta obtener $Z_0 = 50~\Omega$. ¿Qué relación
               $b/a$ necesita?